# 16 — Time-Resolved Decoding Across Datasets

In [ ]:
# Run from the repository root so that relative result paths resolve
# identically regardless of where this notebook lives.
from oddball_eeg.paths import use_project_root

PROJECT_ROOT = use_project_root()

In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression

from oddball_eeg.bids import make_epoching_config_from_experiment
from oddball_eeg.config import get_experiment
from oddball_eeg.io import save_dataframe
from oddball_eeg.loader import load_subjects_grouped
from oddball_eeg.metrics import auc_safe
from oddball_eeg.preprocessing import apply_crop_to_loaded_dataset
from oddball_eeg.splits import build_folds_by_subject
from oddball_eeg.visualization import (
    round_numeric_columns,
    save_figure,
    set_pandas_display_options,
)


set_pandas_display_options()

EXPERIMENT_NAMES = [
    "ds003061_within_subject_max30_wide_timebin",
    "ds005863_within_subject_max30_wide_timebin",
    "ds006466_within_subject_max30_wide_timebin",
    "ds006480_within_subject_max30_wide_timebin",
]

MODEL_NAME = "lda"       # keep this simple first
BIN_MS = 50              # time-resolved bins
OUT_ROOT = Path("results") / "thesis_artifacts" / "time_resolved_decoding"
FIG_DIR = OUT_ROOT / "figures"
TABLE_DIR = OUT_ROOT / "tables"

FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Thesis-ready plotting style
# ------------------------------------------------------------

plt.style.use("default")

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "text.color": "black",
    "axes.labelcolor": "black",
    "axes.edgecolor": "black",
    "xtick.color": "black",
    "ytick.color": "black",
})

In [ ]:
def make_model(model_name: str):
    model_name = str(model_name).lower()

    if model_name == "lda":
        return make_pipeline(
            StandardScaler(),
            LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto"),
        )

    if model_name == "logreg":
        return make_pipeline(
            StandardScaler(),
            LogisticRegression(
                max_iter=2000,
                class_weight="balanced",
                solver="liblinear",
            ),
        )

    raise ValueError(f"Unknown model_name={model_name!r}")


def make_time_bins(times: np.ndarray, bin_ms: int):
    times = np.asarray(times)
    sfreq = 1.0 / float(np.median(np.diff(times)))
    bin_samples = max(1, int(round((bin_ms / 1000.0) * sfreq)))

    n_bins = len(times) // bin_samples
    keep = n_bins * bin_samples

    bins = []
    for b in range(n_bins):
        start = b * bin_samples
        stop = (b + 1) * bin_samples
        idx = np.arange(start, stop)
        bins.append({
            "bin_id": b,
            "start_idx": start,
            "stop_idx": stop,
            "time_start": float(times[start]),
            "time_stop": float(times[stop - 1]),
            "time_center": float(times[idx].mean()),
            "indices": idx,
        })

    return bins


def extract_bin_features(X: np.ndarray, idx: np.ndarray):
    # X shape: trials x channels x time
    # output: trials x channels, averaged inside the bin
    return X[:, :, idx].mean(axis=2)

In [ ]:
all_rows = []

for exp_name in EXPERIMENT_NAMES:
    print("=" * 90)
    print(exp_name)
    print("=" * 90)

    cfg = get_experiment(exp_name)
    epoching_cfg = make_epoching_config_from_experiment(cfg)

    data_full = load_subjects_grouped(
        dataset=cfg.dataset.dataset,
        task_name=cfg.dataset.task,
        subjects=cfg.subjects.subjects,
        epoching_cfg=epoching_cfg,
        chunk_size=cfg.splits.group_chunk_size,
        apply_baseline=True,
        verbose=True,
    )

    crop_window = cfg.effective_crop_window()
    assert crop_window is not None

    data = apply_crop_to_loaded_dataset(
        data_full,
        tmin=crop_window[0],
        tmax=crop_window[1],
    )

    folds_by_sub = build_folds_by_subject(
        data=data,
        n_splits=cfg.splits.n_splits,
        purge_neighbors=cfg.splits.purge_neighbor_groups,
    )

    bins = make_time_bins(data.times, BIN_MS)

    for sub in data.subjects:
        X = data.X_by_sub[sub]
        y = data.y_by_sub[sub]
        folds = folds_by_sub[sub]

        for b in bins:
            idx = b["indices"]

            for fold_id, (tr_idx, te_idx) in enumerate(folds, start=1):
                Xb = extract_bin_features(X, idx)

                X_tr = Xb[tr_idx]
                X_te = Xb[te_idx]
                y_tr = y[tr_idx]
                y_te = y[te_idx]

                clf = make_model(MODEL_NAME)
                clf.fit(X_tr, y_tr)

                if hasattr(clf, "predict_proba"):
                    scores = clf.predict_proba(X_te)[:, 1]
                else:
                    scores = clf.decision_function(X_te)

                auc = auc_safe(y_te, scores)

                all_rows.append({
                    "experiment": exp_name,
                    "dataset": cfg.dataset.dataset,
                    "task": cfg.dataset.task,
                    "event_preset": cfg.dataset.event_preset,
                    "run_tag": cfg.run_tag,
                    "subject": sub,
                    "model": MODEL_NAME,
                    "fold": fold_id,
                    "bin_id": b["bin_id"],
                    "time_start": b["time_start"],
                    "time_stop": b["time_stop"],
                    "time_center": b["time_center"],
                    "auc": auc,
                    "n_train": int(len(tr_idx)),
                    "n_test": int(len(te_idx)),
                    "test_target_rate": float(y_te.mean()),
                })

df_time_resolved = pd.DataFrame(all_rows)
save_dataframe(df_time_resolved, TABLE_DIR / "time_resolved_fold_results.csv")

display(round_numeric_columns(df_time_resolved.head(), 4))

In [ ]:
df_time_summary = (
    df_time_resolved
    .groupby(["experiment", "dataset", "task", "event_preset", "run_tag", "model", "time_center"], as_index=False)
    .agg(
        auc_mean=("auc", "mean"),
        auc_std=("auc", "std"),
        n_rows=("auc", "count"),
    )
    .sort_values(["experiment", "time_center"])
    .reset_index(drop=True)
)

save_dataframe(df_time_summary, TABLE_DIR / "time_resolved_summary.csv")

# ------------------------------------------------------------
# Export clean time-resolved decoding data for final dissertation Figure 6.2
# ------------------------------------------------------------

df_time_export = df_time_summary.copy()

df_time_export = df_time_export.rename(columns={
    "time_center": "time_s",
    "auc_mean": "auc",
    "auc_std": "auc_std",
})

df_time_export["dataset_label"] = df_time_export["dataset"]

df_time_export = df_time_export[
    [
        "experiment",
        "dataset_label",
        "dataset",
        "task",
        "event_preset",
        "run_tag",
        "model",
        "time_s",
        "auc",
        "auc_std",
        "n_rows",
    ]
].copy()

save_dataframe(
    df_time_export,
    TABLE_DIR / "time_resolved_auc_long.csv",
)

display(round_numeric_columns(df_time_export.head(), 4))
print("Saved clean time-resolved data:", TABLE_DIR / "time_resolved_auc_long.csv")

peak_table = (
    df_time_summary
    .sort_values(["experiment", "auc_mean"], ascending=[True, False])
    .groupby("experiment", as_index=False)
    .head(1)
    .reset_index(drop=True)
)

save_dataframe(peak_table, TABLE_DIR / "time_resolved_peak_table.csv")

display(round_numeric_columns(peak_table, 4))

In [ ]:
for exp_name in df_time_summary["experiment"].unique():
    sub_df = (
        df_time_summary[df_time_summary["experiment"] == exp_name]
        .sort_values("time_center")
        .reset_index(drop=True)
    )

    fig, ax = plt.subplots(figsize=(9, 4.5))
    ax.plot(sub_df["time_center"], sub_df["auc_mean"], linewidth=2)

    if "auc_std" in sub_df.columns:
        lo = sub_df["auc_mean"] - sub_df["auc_std"]
        hi = sub_df["auc_mean"] + sub_df["auc_std"]
        ax.fill_between(sub_df["time_center"], lo, hi, alpha=0.15)

    ax.axhline(0.5, linestyle="--", linewidth=1, label="Chance")
    ax.axvline(0.0, linestyle="--", linewidth=1)
    ax.axvspan(0.25, 0.50, alpha=0.12, label="P300 window")

    ax.set_title(f"Time-resolved decoding — {exp_name}")
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("ROC-AUC")
    ax.set_ylim(0.0, 1.0)
    ax.legend()
    ax.grid(True, alpha=0.3)

    fig.tight_layout()
    save_figure(fig, FIG_DIR / f"{exp_name}_time_resolved_auc.png")
    plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

for exp_name in df_time_summary["experiment"].unique():
    sub_df = (
        df_time_summary[df_time_summary["experiment"] == exp_name]
        .sort_values("time_center")
        .reset_index(drop=True)
    )
    ax.plot(sub_df["time_center"], sub_df["auc_mean"], linewidth=2, label=exp_name)

ax.axhline(0.5, linestyle="--", linewidth=1)
ax.axvline(0.0, linestyle="--", linewidth=1)
ax.axvspan(0.25, 0.50, alpha=0.10)

ax.set_title("Time-resolved decoding comparison across datasets")
ax.set_xlabel("Time (s)")
ax.set_ylabel("ROC-AUC")
ax.set_ylim(0.0, 1.0)
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

fig.tight_layout()
save_figure(fig, FIG_DIR / "time_resolved_auc_all_datasets.png")
plt.show()